# 01 — Bronze Ingestion
Loads the temporary PGCB Excel source into a raw Delta table.

**Input:** `/Volumes/pgcb/raw/files/PGCB_date_power_demand.xlsx`  
**Output:** `pgcb.bronze.grid_raw`

The Bronze layer intentionally performs only minimal source handling and adds ingestion metadata. Data-quality corrections belong in the Silver layer.

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import functions as F

SOURCE_FILE = "/Volumes/pgcb/raw/files/PGCB_date_power_demand.xlsx"
BRONZE_TABLE = "pgcb.bronze.grid_raw"
SOURCE_NAME = "PGCB_date_power_demand.xlsx"

## Read source file

In [ ]:
pdf = pd.read_excel(SOURCE_FILE)

print(f"Source rows: {len(pdf):,}")
print(f"Source columns: {len(pdf.columns)}")
display(pdf.head())

## Add ingestion metadata and convert to Spark

In [ ]:
pdf["_source_file"] = SOURCE_NAME
pdf["_ingested_at"] = datetime.now(timezone.utc)

spark_df = spark.createDataFrame(pdf)
spark_df.printSchema()

## Write Bronze Delta table

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS pgcb.bronze")

(
    spark_df.write
    .format("delta")
    .mode("overwrite")  # Historical bootstrap. Use append/MERGE for future incremental scraping.
    .saveAsTable(BRONZE_TABLE)
)

## Bronze validation

In [ ]:
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze rows: {bronze_df.count():,}")
display(bronze_df.limit(10))

null_counts = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
])
display(null_counts)

duplicate_timestamp_count = (
    bronze_df.groupBy("datetime")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Timestamps appearing more than once: {duplicate_timestamp_count:,}")